In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
import matplotlib.pyplot as plt
# from data_preparation.field import load_fields

In [ ]:
processed_data_dir = Path("../processed_data")
exp_name = "history1_fv"
case_name = "flange"
case_name = f"{case_name}_{exp_name}"
case_dir = processed_data_dir / case_name
checkpoint_dir = case_dir / "checkpoints"
pred_dir = case_dir / "predictions"
error_dir = case_dir / "errors"

In [ ]:
train_loss = np.load(os.path.join(checkpoint_dir, "train_losses.npy"))
val_loss = np.load(os.path.join(checkpoint_dir, "val_losses.npy"))
rollout_mae = np.load(os.path.join(checkpoint_dir, "rollout_mae.npy"))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

loss_graph, ax = plt.subplots(figsize=(960*px, 540*px))

# Transparent background
loss_graph.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.title.set_color(AX_COLOR)
ax.xaxis.label.set_color(AX_COLOR)
ax.yaxis.label.set_color(AX_COLOR)
ax.tick_params(colors=AX_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor(AX_COLOR)

plt.plot(train_loss, label="Train Loss", color = PLOT1_COLOR, linewidth=2.5)
plt.plot(val_loss, label="Validation Loss", color = PLOT2_COLOR, linewidth=2.5)
ax.set(yscale='log')

legend = ax.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0–1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

plt.xlabel("Epoch")
plt.ylabel("Loss")
ax.set_title("Training and Validation Loss, h = 1, FV", fontsize=24)
ax.set_yscale("log")
ax.set_xlabel("Epoch", fontsize=18)
ax.set_ylabel("MSE Loss", fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


# plt.savefig(f"../outputs/plots/{case_name}.png", dpi=300, transparent = True)

In [ ]:
train_loss, val_loss, rollout_mae = {},{},{}
processed_data_dir = Path("../processed_data")
for case_name in ["flange_history1_fv", "flange_history1_nofv"]:
    case_dir = processed_data_dir / case_name
    checkpoint_dir = case_dir / "checkpoints"
    pred_dir = case_dir / "predictions"
    error_dir = case_dir / "errors"

    train_loss[case_name] = np.load(os.path.join(checkpoint_dir, "train_losses.npy"))
    val_loss[case_name] = np.load(os.path.join(checkpoint_dir, "val_losses.npy"))
    rollout_mae[case_name] = np.load(os.path.join(checkpoint_dir, "rollout_mae.npy"))

In [ ]:
fig, axs = plt.subplots(1,1)
axs.set_title("Training Loss", fontsize=20)
axs.set_yscale("log")
axs.set_xlabel("Epoch", fontsize=18)
axs.set_ylabel("MSE Loss", fontsize=18)
for i,case_name in enumerate(train_loss):
    plt.plot(train_loss[case_name], label=f"{case_name}: Train Loss")
    plt.plot(val_loss[case_name], label=f"{case_name}: Validation Loss")

axs.legend(fontsize=14)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(1,1, figsize=(960*px, 540*px))


# Transparent background
loss_graph.patch.set_alpha(0)
axs.patch.set_alpha(0)

axs.title.set_color(AX_COLOR)
axs.xaxis.label.set_color(AX_COLOR)
axs.yaxis.label.set_color(AX_COLOR)
axs.tick_params(colors=AX_COLOR)
for spine in axs.spines.values():
    spine.set_edgecolor(AX_COLOR)


for i,case_name in enumerate(train_loss):
    if case_name.endswith("_fv"):
        plt.plot(rollout_mae[case_name], label="FV", color = PLOT1_COLOR, linewidth=2.5)
    elif case_name.endswith("_nofv"):
        plt.plot(rollout_mae[case_name], label="No FV", color = PLOT2_COLOR, linewidth=2.5, linestyle='--')

legend = axs.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0–1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

axs.set_title("Rollout Error Accumulation, h = 1", fontsize=24)
axs.set_xlabel("Time step", fontsize=18)
axs.set_ylabel("MAE", fontsize=18)
axs.tick_params(axis='both', labelsize=16)
axs.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig(f"../outputs/plots/{case_name[:15]}_rollout.png", dpi=300, transparent = True)


## Parametric case

Training/validation-loss and rollout-error plots for the parametric runs at `history = 1`.
The rollout panels mirror the flange **FV vs no-FV** comparison
(`parametric_history1_mesh_correct_edge_attr` vs `parametric_history1_mesh_nofv`),
one panel per mesh held out from those checkpoints: `model_003` and `model_007`
(the original 10-mesh study's test set) plus `model_010` and `model_011`
(the 2-hole / 3-hole geometries added later, which postdate both checkpoints).

In [ ]:
# --- Parametric case (history = 1, FV vs no-FV) ------------------------
# Each parametric run is tested on the held-out meshes model_003 and
# model_007, so there is one rollout-MAE curve per test mesh.
param_cases = {
    "fv":   "parametric_history1_mesh_correct_edge_attr",
    "nofv": "parametric_history1_mesh_nofv",
}
param_test_meshes = ["model_003", "model_007", "model_010", "model_011"]

# The FV run drives the single-case loss plot below.
param_case = param_cases["fv"]
param_ckpt = processed_data_dir / param_case / "checkpoints"
param_train_loss = np.load(os.path.join(param_ckpt, "train_losses.npy"))
param_val_loss = np.load(os.path.join(param_ckpt, "val_losses.npy"))

# rollout MAE per {fv/nofv} run and per test mesh.
param_rollout = {}
for key, cname in param_cases.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    param_rollout[key] = {
        mesh: np.load(os.path.join(ckpt, f"rollout_mae_{mesh}.npy"))
        for mesh in param_test_meshes
    }


In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

loss_graph, ax = plt.subplots(figsize=(960*px, 540*px))

# Transparent background
loss_graph.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.title.set_color(AX_COLOR)
ax.xaxis.label.set_color(AX_COLOR)
ax.yaxis.label.set_color(AX_COLOR)
ax.tick_params(colors=AX_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor(AX_COLOR)

plt.plot(param_train_loss, label="Train Loss", color = PLOT1_COLOR, linewidth=2.5)
plt.plot(param_val_loss, label="Validation Loss", color = PLOT2_COLOR, linewidth=2.5)
ax.set(yscale='log')

legend = ax.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

plt.xlabel("Epoch")
plt.ylabel("Loss")
ax.set_title("Training and Validation Loss, Parametric FV, h = 1", fontsize=24)
ax.set_yscale("log")
ax.set_xlabel("Epoch", fontsize=18)
ax.set_ylabel("MSE Loss", fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


# plt.savefig(f"../outputs/plots/{param_case}.png", dpi=300, transparent = True)


In [ ]:
from matplotlib.ticker import StrMethodFormatter

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# One panel per held-out test mesh; each panel is the flange-style
# FV vs no-FV rollout comparison.
fig, axs = plt.subplots(len(param_test_meshes),1,
                        figsize=(960*px, len(param_test_meshes)*540*px),
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)

for ax, mesh in zip(axs, param_test_meshes[::-1]):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    fv_curve = param_rollout["fv"][mesh]
    nofv_curve = param_rollout["nofv"][mesh]
    ax.plot(fv_curve, label="FV",
            color=PLOT1_COLOR, linewidth=2.5)
    ax.plot(nofv_curve, label="No FV",
            color=PLOT2_COLOR, linewidth=2.5, linestyle='--')

    # Log-scale whenever the two curves span more than ~1.5 decades, otherwise
    # the FV curve is squashed onto the axis by the diverging no-FV one. Was
    # hardcoded to model_003; made data-driven so added meshes are handled too.
    both = np.concatenate([fv_curve, nofv_curve])
    if both.max() / max(both.min(), 1e-30) > 30:
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.1e}'))

    legend = ax.legend(fontsize=18)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(f"Rollout Error Accumulation, {mesh}, h = 1", fontsize=24)
    ax.set_xlabel("Time step", fontsize=18)
    ax.set_ylabel("MAE", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.tight_layout(pad=0)
# plt.savefig("../outputs/plots/parametric_history1_mesh_rollout_exp.png", dpi=300, transparent = True)

## Direct vs. residual prediction

The same parametric FV model (`history = 1`, one MP layer), producing $T^{n+1}$ two ways:

- **Direct** (`parametric_history1_mesh_correct_edge_attr`): the network outputs the
  normalized $T^{n+1}$.
- **Residual** (`parametric_history1_mesh_correct_edge_attr_residual`): the network
  outputs the increment and `FVSurrogate` adds it back,
  $T^{n+1} = T^n + s\,f(\cdot)$ with $s = \mathrm{rms}(\Delta T)/\sigma_T$ fitted on the
  training meshes.

Both train on the same six meshes, validate on `model_000` / `model_009` and share the
normalizer, and in both the loss is the MSE on the normalized $T^{n+1}$, so the loss
curves compare directly. The direct run trained for 500 epochs and the residual one for
200, so the loss panels stop at epoch 200; the rollout panels use each run's final
checkpoint on the four held-out meshes.

In [ ]:
# --- Direct vs residual prediction (parametric, FV, history = 1) ----------
# Both runs share the train/val meshes and the normalizer, and both losses are
# the MSE on the normalized T^{n+1}, so their curves compare directly.
import json

resid_cases = {
    "direct":   "parametric_history1_mesh_correct_edge_attr",
    "residual": "parametric_history1_mesh_correct_edge_attr_residual",
}
resid_test_meshes = ["model_003", "model_007", "model_010", "model_011"]
# The residual run trained for 200 epochs and the direct one for 500: compare
# the losses over the window both have.
N_EPOCHS_CMP = 200

resid_train_loss, resid_val_loss, resid_rollout, resid_split = {}, {}, {}, {}
for key, cname in resid_cases.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    resid_train_loss[key] = np.load(ckpt / "train_losses.npy")[:N_EPOCHS_CMP]
    resid_val_loss[key] = np.load(ckpt / "val_losses.npy")[:N_EPOCHS_CMP]
    resid_rollout[key] = {mesh: np.load(ckpt / f"rollout_mae_{mesh}.npy")
                          for mesh in resid_test_meshes}
    with open(ckpt / "mesh_split.json") as fh:
        resid_split[key] = json.load(fh)

# The comparison only means something if both runs saw the same meshes.
for part in ("train", "val"):
    assert sorted(resid_split["direct"][part]) == sorted(resid_split["residual"][part]), \
        f"{part} meshes differ: {resid_split['direct'][part]} vs {resid_split['residual'][part]}"

print(f"{'':10s}  {'rollout MAE, mean [K]':>22s}  {'rollout MAE, last step [K]':>27s}")
print(f"{'mesh':10s}  {'direct':>10s} {'residual':>11s}  {'direct':>15s} {'residual':>11s}")
for mesh in resid_test_meshes:
    d, r = resid_rollout["direct"][mesh], resid_rollout["residual"][mesh]
    print(f"{mesh:10s}  {d.mean():10.3f} {r.mean():11.3f}  {d[-1]:15.3f} {r[-1]:11.3f}")
print(f"\nbest validation loss in the first {N_EPOCHS_CMP} epochs: " + ", ".join(
    f"{key} {v.min():.3e} (epoch {v.argmin()})" for key, v in resid_val_loss.items()))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Residual is the variant under test (solid, PLOT1) and direct the baseline
# (dashed, PLOT2), the roles FV / no-FV take in the rollout panels above.
# Shared y-axis, so the training and validation panels read on one scale.
fig, axs = plt.subplots(1, 2, figsize=(1400*px, 540*px), sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle(f"Direct vs Residual Loss, Parametric FV, h = 1 (first {N_EPOCHS_CMP} epochs)",
             fontsize=24, color=AX_COLOR)

for ax, (title, losses) in zip(axs, (("Training Loss", resid_train_loss),
                                     ("Validation Loss", resid_val_loss))):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    ax.plot(losses["residual"], label="Residual", color=PLOT1_COLOR, linewidth=2.5)
    ax.plot(losses["direct"], label="Direct", color=PLOT2_COLOR, linewidth=2.5, linestyle='--')

    # Value where each curve ends: the higher one labelled above its line and
    # the lower one below, so the two labels cannot collide.
    ends = sorted(((losses[k][-1], k) for k in ("residual", "direct")), reverse=True)
    for (y, key), dy, va in zip(ends, (8, -8), ("bottom", "top")):
        ax.annotate(f"{y:.1e}", xy=(len(losses[key]) - 1, y), xytext=(0, dy),
                    textcoords="offset points", ha="right", va=va,
                    fontsize=14, color=AX_COLOR)

    ax.set_yscale("log")

    legend = ax.legend(fontsize=18)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if ax is axs[0]:
        ax.set_ylabel("MSE Loss", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_direct_vs_residual_loss.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# One panel per held-out mesh, each run at its final checkpoint; same roles and
# line styles as the loss panels.
fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Rollout Error Accumulation, Direct vs Residual, h = 1",
             fontsize=24, color=AX_COLOR)

for ax, mesh in zip(axs.ravel(), resid_test_meshes):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    ax.plot(resid_rollout["residual"][mesh], label="Residual",
            color=PLOT1_COLOR, linewidth=2.5)
    ax.plot(resid_rollout["direct"][mesh], label="Direct",
            color=PLOT2_COLOR, linewidth=2.5, linestyle='--')
    ax.set_ylim(bottom=0)

    legend = ax.legend(fontsize=15)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(mesh, fontsize=19)
    ax.set_xlabel("Time step", fontsize=15)
    ax.set_ylabel("MAE [K]", fontsize=15)
    ax.tick_params(axis='both', labelsize=13)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_direct_vs_residual_rollout.png", dpi=300, transparent = True)

## Message vs. finite-volume flux

`FiniteVolumeGraphNet` sums its messages at the receiving node the way a finite-volume
scheme sums face fluxes over a cell, so the obvious question is whether the learned
$m_{ij}$ *is* the flux `laplacianFoam` assembles for `laplacian(DT,T)` with
`Gauss linear corrected`:

$$F_{j\rightarrow i} \;=\; D_T\,|S_f|\left[\;\frac{T_j - T_i}{|d|\,\max(\cos,\,0.05)} \;+\; k_f\cdot(\nabla T)_f\;\right]$$

Everything below runs on the **parametric** study — the same two `history = 1`
checkpoints the rollout panels compare (`parametric_history1_mesh_correct_edge_attr`
vs `parametric_history1_mesh_nofv`), probed on the same four held-out meshes:
`model_003` and `model_007` from the recorded split, plus the later `model_010` /
`model_011`. Nothing in the split is temporal, so every timestep of those meshes is
test data and the probe is free to sample the whole sequence.

`propagate` throws $m_{ij}$ away before `forward` returns, so it is pulled back out with
`models.message_probe.MessageProbe`, which hangs a PyG `register_message_forward_hook` on
each MP layer. Nothing in `fvgnn.py` or the `state_dict` changes, so every checkpoint
already on disk is probed as-is:

```python
probe = MessageProbe(model)
with probe, torch.no_grad():
    for i in range(n): model(data_i)

probe.messages[0]            # (E, 128)    last pass, layer 0
probe.stacked_messages(0)    # (n, E, 128) whole rollout
probe.stacked_aggregated(0)  # (n, N, 128) sum_j m_ij
```

`analyze_messages.py` runs the same comparison from the command line.

### Orientation — the thing that silently breaks this

PyG aggregates at `edge_index[1]`, so inside `message()` **`x_i` is the receiver and
`x_j` the sender**, and `build_static_graph` already orients each edge's stored $S_f$
*into* the receiver. `fv_flux` follows the same convention: a positive flux heats the
receiver, and summing over the in-edges of node $i$ gives $V_i\,\mathrm{d}T_i/\mathrm{d}t$.
Get this backwards and every correlation below flips sign while still looking plausible.

In [ ]:
import sys
print(sys.executable)
print(torch.__file__)

In [ ]:
# --- Message vs FV flux: extract, and cache -------------------------------
# Runs both trained parametric models over N_STEPS timesteps of every held-out
# mesh with a MessageProbe attached, and stores everything the figures below
# need. A parametric mesh carries up to ~270k edges, so a full (steps, E, 128)
# message stack is several GB: only a random subset of whole faces is held in
# memory, and only the reduced statistics reach the cache.

import torch

from data_preparation.field import load_fields
from data_preparation.mesh_dataset import SingleMeshDataset
from data_preparation.normalization import FeatureNormalizer
from data_preparation.static_graph import build_static_graph
from mesh2graph.utils import filter_of_time_directories
from models.fvgnn import FVSurrogate
from models.message_probe import (MessageProbe, fv_flux, fv_flux_corrected,
                                  load_cell_gradient, paired_edge_index)

FLUX_EXPS   = {"FV": "parametric_history1_mesh_correct_edge_attr",
               "No FV": "parametric_history1_mesh_nofv"}
# The same held-out meshes the rollout panels above compare.
FLUX_MESHES = ["model_003", "model_007", "model_010", "model_011"]
EXCLUDED    = ["top", "bottom", "cbores"]
DT_PARAM    = 4e-5          # constant/transportProperties
HISTORY     = 1
# The mesh split is over GEOMETRY, not time, so every timestep of a held-out
# mesh is test data. These 12 are spread over the whole sequence rather than
# taken consecutively, so the transient does not dominate the statistics.
N_STEPS     = 12
KEEP_FACES  = 30_000        # faces (= 2 opposite edges) kept in memory per mesh
N_SCATTER   = 40_000        # points kept for the density panels
REF_STEPS   = list(range(1, 41))   # window the reference flux is validated on

raw_param_dir = Path("../raw_data/parametric")
flux_cache = processed_data_dir / "parametric_message_flux.npz"


def _r2(X, y):
    """R^2 of the least-squares fit y ~ [X, 1]."""
    X, y = X.double(), y.double()
    X = torch.cat([X, torch.ones(X.shape[0], 1, dtype=X.dtype)], dim=1)
    beta = torch.linalg.lstsq(X, y[:, None]).solution
    ss_res = ((y[:, None] - X @ beta) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return float(1.0 - ss_res / ss_tot.clamp(min=1e-30))


def load_mesh(name):
    """(static_graph, T_sequence) for one parametric case, cached to disk.

    Parsing an OpenFOAM case is by far the slowest step here, and the raw
    geometry is the same for both checkpoints, so it is parsed once per mesh.
    """
    cache = (processed_data_dir / "parametric_preproc_cache"
             / f"{name}__T__{'-'.join(EXCLUDED)}.pt")
    if cache.exists():
        blob = torch.load(cache, weights_only=False)
        return blob["graph"], blob["T"]
    g = build_static_graph(str(raw_param_dir / name), EXCLUDED)
    T = load_fields(str(raw_param_dir / name), "T", excluded_patches=EXCLUDED)
    cache.parent.mkdir(exist_ok=True)
    torch.save({"graph": g, "T": T}, cache)
    return g, T


def compute_flux_stats():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in FLUX_MESHES:
        graph_full, T_seq = load_mesh(mesh)
        case_dir = str(raw_param_dir / mesh)
        times = filter_of_time_directories(case_dir)

        ei, ea = graph_full.edge_index, graph_full.edge_attr   # RAW geometry
        N, E = graph_full.num_nodes, ei.shape[1]
        n_int = int((graph_full.node_attr[:, 0] == 1.0).sum())
        rev = paired_edge_index(ei)
        internal = (ei[0] < n_int) & (ei[1] < n_int)
        cos = ea[:, 9].double()

        # -- how trustworthy is the reference? Fit V_i per cell in
        #    V_i dT_i/dt = sum_j F_ij and count the impossible (V_i < 0) fits,
        #    for the orthogonal flux and for the non-orth-corrected one.
        dt = float(times[2]) - float(times[1])
        gradT_ref = load_cell_gradient(case_dir, [times[i] for i in REF_STEPS])
        dTdt = (T_seq[[i + 1 for i in REF_STEPS]].double()
                - T_seq[[i - 1 for i in REF_STEPS]].double()) / (2 * dt)
        worst = torch.full((n_int,), 2.0, dtype=torch.double)   # worst face cos per cell
        into_cell = ei[1] < n_int
        worst.scatter_reduce_(0, ei[1][into_cell], cos[into_cell], reduce="amin")
        for tag, F_ref in (
                ("orth", fv_flux(ei, ea, T_seq[REF_STEPS].double(), DT=DT_PARAM)),
                ("corr", fv_flux_corrected(ei, ea, T_seq[REF_STEPS].double(),
                                           gradT_ref, DT=DT_PARAM,
                                           n_internal_nodes=n_int))):
            Fn = torch.zeros(len(REF_STEPS), N, dtype=torch.double)
            Fn.index_add_(1, ei[1], F_ref)
            Fn, d = Fn[:, :n_int], dTdt[:, :n_int]
            V = (d * Fn).sum(0) / (d * d).sum(0).clamp(min=1e-300)
            out[f"ref__{mesh}__{tag}_r2"] = float(
                1 - ((Fn - V[None] * d) ** 2).sum() / (Fn - Fn.mean(0)).pow(2).sum())
            out[f"ref__{mesh}__{tag}_negV"] = float((V < 0).double().mean())
            out[f"ref__{mesh}__{tag}_bins"] = np.array(
                [[int(sel.sum()), float((V[sel] < 0).double().mean())]
                 for sel in ((worst > lo) & (worst <= hi)
                             for lo, hi in ((0.999, 2.0), (0.99, 0.999),
                                            (0.95, 0.99), (-1.0, 0.95)))])

        # -- edges held in memory: whole faces, both halves internal, so that
        #    the antisymmetry test below has both m_ij and m_ji.
        canon = torch.nonzero((rev > torch.arange(E)) & internal
                              & internal[rev]).squeeze(1)
        n_faces = min(KEEP_FACES, canon.numel())
        sel = canon[torch.randperm(canon.numel(),
                                   generator=torch.Generator().manual_seed(0))[:n_faces]]
        kept = torch.cat([sel, rev[sel]])
        rev_local = torch.cat([torch.arange(n_faces) + n_faces,
                               torch.arange(n_faces)])
        near = cos[kept] > 0.99
        out[f"{mesh}__n_edges"] = E
        out[f"{mesh}__n_internal"] = int(internal.sum())
        out[f"{mesh}__n_kept"] = int(kept.numel())
        out[f"{mesh}__n_kept_near"] = int(near.sum())

        # -- the reference flux over the probed timesteps
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()
        gradT = load_cell_gradient(case_dir, [times[i] for i in steps])
        T_at = T_seq[steps].double()
        F = fv_flux_corrected(ei, ea, T_at, gradT, DT=DT_PARAM,
                              n_internal_nodes=n_int)            # (S, E)
        F_node = torch.zeros(N_STEPS, N, dtype=torch.double)
        F_node.index_add_(1, ei[1], F)                           # (S, N)
        # F is w*dT plus the correction, so regressing on each factor separately
        # says WHICH one the message picked up.
        tgts = {"F": F[:, kept].reshape(-1),
                "dT": (T_at[:, ei[0]] - T_at[:, ei[1]])[:, kept].reshape(-1),
                "w": (DT_PARAM * ea[:, 7].double()
                      / (ea[:, 3].double() * cos.clamp(min=0.05)))[kept].repeat(N_STEPS)}

        for label, exp in FLUX_EXPS.items():
            graph = graph_full.clone()
            if exp.endswith("_nofv"):
                graph.edge_attr = graph.edge_attr[:, :4]
            # The normalizer was fitted on the TRAINING meshes and saved with the
            # checkpoint; refitting it here would leak the held-out geometry.
            norm = FeatureNormalizer()
            norm.load(processed_data_dir / exp / "checkpoints" / "normalizer.pt")
            ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)

            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                hidden_dim=64, out_dim=1, n_mp_layers=1,
            ).to(device)
            model.load_state_dict(torch.load(
                processed_data_dir / exp / "checkpoints" / "model.pt",
                map_location=device))
            model.eval()

            # history=1, so sample i is built from T[i] — the field the flux
            # must be evaluated on. Reduce every step immediately: the full
            # (E, 128) message is dropped as soon as `kept` is sliced out of it.
            msg_kept, aggr_all = [], []
            probe = MessageProbe(model)
            with probe, torch.no_grad():
                for i in steps:
                    model(ds[i].to(device))
                    msg_kept.append(probe.messages[0][kept])
                    aggr_all.append(probe.aggregated[0])
                    probe.clear()
            msg = torch.stack(msg_kept)                 # (S, kept, C)
            aggr = torch.stack(aggr_all)                # (S, N, C)

            # antisymmetric energy fraction of m_ij across the two halves of a face
            S_, A_ = 0.5 * (msg + msg[:, rev_local]), 0.5 * (msg - msg[:, rev_local])
            sa = A_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
            ss = S_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
            out[f"{exp}__{mesh}__anti"] = (sa / (sa + ss)).numpy()      # (C,)

            # linear read-out of each target from the full 128-channel message
            m_flat = msg.reshape(-1, msg.shape[-1]).double()
            out[f"{exp}__{mesh}__r2"] = np.array(
                [_r2(m_flat, t) for t in tgts.values()])
            # control: the near-orthogonal edges, where even the uncorrected
            # flux is exact and the reference cannot be blamed.
            nf = near.repeat(N_STEPS)
            out[f"{exp}__{mesh}__r2_near"] = np.array(
                [_r2(m_flat[nf], t[nf]) for t in tgts.values()])
            # node level: sum_j m_ij vs the net flux that drives dT_i/dt
            out[f"{exp}__{mesh}__r2_node"] = _r2(
                aggr[:, :n_int].reshape(-1, msg.shape[-1]).double(),
                F_node[:, :n_int].reshape(-1))

            # best-correlated channel, and a subsample for the density panels
            mc = m_flat - m_flat.mean(0, keepdim=True)
            Fc = tgts["F"] - tgts["F"].mean()
            r = (mc * Fc[:, None]).sum(0) / (mc.norm(dim=0) * Fc.norm()).clamp(min=1e-30)
            best = int(r.abs().argmax())
            pick = torch.randperm(m_flat.shape[0],
                                  generator=torch.Generator().manual_seed(0))[:N_SCATTER]
            out[f"{exp}__{mesh}__best_ch"] = best
            out[f"{exp}__{mesh}__best_r"] = float(r[best])
            out[f"{exp}__{mesh}__sc_m"] = m_flat[pick, best].float().numpy()
            out[f"{exp}__{mesh}__sc_F"] = tgts["F"][pick].float().numpy()
            out[f"{exp}__{mesh}__sc_dT"] = tgts["dT"][pick].float().numpy()
            del msg, aggr, m_flat, probe, model
    return out


if flux_cache.exists():
    flux = dict(np.load(flux_cache, allow_pickle=False))
else:
    flux = compute_flux_stats()
    np.savez_compressed(flux_cache, **flux)

print(f"{'mesh':<11}{'model':<7}{'ch':>4}{'r':>8}{'R2 F':>7}{'R2 dT':>7}{'R2 w':>7}"
      f"{'node':>7}{'anti':>7}   R2 F/dT/w on cos>0.99")
for mesh in FLUX_MESHES:
    print(f"{mesh}  reference: V_i fit R^2 "
          f"{float(flux[f'ref__{mesh}__orth_r2']):.3f} (orthogonal) -> "
          f"{float(flux[f'ref__{mesh}__corr_r2']):.3f} (corrected), V_i < 0 on "
          f"{100*float(flux[f'ref__{mesh}__orth_negV']):.0f}% -> "
          f"{100*float(flux[f'ref__{mesh}__corr_negV']):.0f}% of cells")
    for label, exp in FLUX_EXPS.items():
        print(f"{mesh:<11}{label:<7}{int(flux[f'{exp}__{mesh}__best_ch']):>4}"
              f"{float(flux[f'{exp}__{mesh}__best_r']):>+8.3f}"
              + "".join(f"{v:>7.3f}" for v in flux[f"{exp}__{mesh}__r2"])
              + f"{float(flux[f'{exp}__{mesh}__r2_node']):>7.3f}"
              f"{flux[f'{exp}__{mesh}__anti'].mean():>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux[f"{exp}__{mesh}__r2_near"]))

### Is the reference trustworthy?

Checked against the actual `laplacianFoam` solution before reading anything into the
comparison. The geometry is exact: every edge has its reverse partner,
$S_{f,ij} = -S_{f,ji}$ to machine zero, and $\cos(S_f, d) > 0$ everywhere — internal
minimum 0.63 on `model_003` and 0.74 on `model_011`, down to 0.11 on the boundary edges,
whose column `build_static_graph` recomputes instead of leaving it at OpenFOAM's
hardcoded 1.0.

**The orthogonal flux alone is not good enough on these meshes.** Fitting $V_i$ per cell
in $V_i\,\mathrm{d}T_i/\mathrm{d}t = \sum_j F_{ij}$ over $t = 0.025\ldots1.0$ and counting
the physically impossible negative fits, pooled over the four held-out meshes:

| worst face $\cos$ of cell | cells | $V_i < 0$, orthogonal | $V_i < 0$, corrected |
|---|---|---|---|
| $> 0.999$ | 5803 | 4.2 % | 4.1 % |
| $0.99 - 0.999$ | 22544 | 5.4 % | 2.7 % |
| $0.95 - 0.99$ | 29157 | 21.3 % | 1.7 % |
| $< 0.95$ | 52546 | 25.7 % | 2.3 % |

These are `snappyHexMesh` geometries: only about a quarter of the cells have all their
faces above $\cos = 0.99$, so the flange's fix — restricting to near-orthogonal edges —
would throw most of the mesh away here. Instead the explicit non-orthogonal correction
$k_f\cdot(\nabla T)_f$ is added back with OpenFOAM's **own** cell gradient, which
`laplacianFoam` writes at every write time (`gradTx/y/z`), via
`message_probe.fv_flux_corrected`. That closes the balance to $R^2 = 0.995 - 0.996$ per
mesh, against $0.79 - 0.84$ uncorrected, and drops the impossible fits from ~20 % of
cells to 0.1 - 4.7 %.

So everything below runs on **all internal edges**, not just the orthogonal ones. Two
approximations survive in the correction — the face gradient is a 0.5/0.5 midpoint rather
than OpenFOAM's `linear` weights, and the correction is applied on internal faces only —
and they are what the residual few percent of bad cells are. The near-orthogonal control
($\cos > 0.99$, about 43k of the 60k edges kept per mesh) is computed alongside and
raises every $R^2$ below by 0.01 - 0.10. It changes no conclusion — the only FV vs no-FV
comparison it flips is the near-tied $\Delta T$ on `model_007` — so nothing here rests on
the correction being perfect.

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)

bins = np.linspace(0, 1, 31)
for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for (label, exp), color, ls in zip(FLUX_EXPS.items(),
                                       (PLOT1_COLOR, PLOT2_COLOR), ("-", "--")):
        ax.hist(flux[f"{exp}__{mesh}__anti"], bins=bins, histtype="step",
                linewidth=2.5, color=color, linestyle=ls, label=label)

    # The FV flux sits exactly at 1.0 — that is what makes the scheme conservative.
    ax.axvline(1.0, color=AX_COLOR, linewidth=2.5)
    ax.annotate("FV flux = 1.0\n(conservative)", xy=(1.0, 0.97),
                xycoords=("data", "axes fraction"), xytext=(-10, 0),
                textcoords="offset points", ha="right", va="top",
                fontsize=13, color=AX_COLOR)

    legend = ax.legend(fontsize=15, loc="upper center")
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_xlim(0, 1.05)
    ax.set_title(mesh, fontsize=19)
    ax.set_xlabel(r"Antisymmetric energy fraction  $\|A\|^2/(\|S\|^2+\|A\|^2)$",
                  fontsize=15)
    ax.set_ylabel("Message channels", fontsize=15)
    ax.grid(True, which="both", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)

fig.suptitle("Is the message antisymmetric across a face?",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_antisymmetry.png", dpi=300, transparent=True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)

# F is w*dT plus the non-orthogonal correction, so regressing on each factor
# separately says WHICH one the message picked up.
targets = [r"$F$", r"$\Delta T = T_j - T_i$", r"$w = D_T|S_f|/(|d|\cos)$"]
x = np.arange(len(targets))
width = 0.28

for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, ((label, exp), color) in enumerate(zip(FLUX_EXPS.items(),
                                                  (PLOT1_COLOR, PLOT2_COLOR))):
        vals = flux[f"{exp}__{mesh}__r2"]
        off = (i - 0.5) * (width + 0.02)      # 2% surface gap between the pair
        bars = ax.bar(x + off, vals, width, color=color, label=label)
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=13, color=AX_COLOR)

    legend = ax.legend(fontsize=15)
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 0.82)
    ax.set_title(f"{mesh}  —  {int(flux[f'{mesh}__n_kept'])//1000}k of "
                 f"{int(flux[f'{mesh}__n_internal'])//1000}k internal edges",
                 fontsize=17)
    ax.set_ylabel(r"$R^2$  (all 128 channels $\rightarrow$ target)", fontsize=15)
    ax.grid(True, axis="y", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
    ax.set_axisbelow(True)

fig.suptitle(f"What the message linearly encodes  ({N_STEPS} probed timesteps)",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_r2.png", dpi=300, transparent=True)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Sequential ramp for point density: one hue, light -> dark.
DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "fvgn_density", ["#f0eee9", "#b9a8cb", PLOT1_COLOR, "#2b1a3c"])

# Columns: held-out meshes. Rows: the flux, and the bare temperature difference
# that is one of its two factors.
fig, axs = plt.subplots(2, len(FLUX_MESHES),
                        figsize=(1600*px, 800*px), constrained_layout=True)
fig.patch.set_alpha(0)

exp = FLUX_EXPS["FV"]
for col, mesh in enumerate(FLUX_MESHES):
    ch = int(flux[f"{exp}__{mesh}__best_ch"])
    m = flux[f"{exp}__{mesh}__sc_m"]
    panels = [("$F$  [K m$^3$ s$^{-1}$]", flux[f"{exp}__{mesh}__sc_F"]),
              (r"$\Delta T$  [K]", flux[f"{exp}__{mesh}__sc_dT"])]

    for row, (xlabel, xv) in enumerate(panels):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        hb = ax.hexbin(xv, m, gridsize=50, bins="log", cmap=DENSITY_CMAP,
                       mincnt=1, linewidths=0)
        r = np.corrcoef(xv, m)[0, 1]
        if row == 0:
            ax.set_title(f"{mesh} · channel {ch}", fontsize=17)
        ax.annotate(f"$r = {r:+.2f}$", xy=(0.03, 0.95), xycoords="axes fraction",
                    va="top", fontsize=15, color=AX_COLOR)
        ax.set_xlabel(xlabel, fontsize=15)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 3))
        ax.grid(True, ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
        ax.set_axisbelow(True)
        if col == 0:
            ax.set_ylabel("message value", fontsize=15)

cb = fig.colorbar(hb, ax=axs, pad=0.02)
cb.set_label("edge-samples per bin", fontsize=14, color=AX_COLOR)
cb.ax.tick_params(colors=AX_COLOR, labelsize=12)
cb.outline.set_edgecolor(AX_COLOR)

fig.suptitle("Most flux-correlated channel of the FV model",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_density.png", dpi=300, transparent=True)

### What comes out

**The message is not a conservative flux.** A true FV flux is exactly antisymmetric
across a face, $F_{ij} = -F_{ji}$ — that is what makes the scheme conservative — so its
antisymmetric energy fraction is 1.0. The learned message averages **0.17 - 0.28** across
channels on all four held-out meshes: roughly three quarters of its energy is
*symmetric*, the opposite of a flux. The most antisymmetric single channel reaches only
0.50 - 0.62 (FV) and 0.76 - 0.81 (no-FV). The no-FV message is consistently the more
antisymmetric of the two — 9 to 20 channels above 0.5, against 0 to 5 for the FV model —
which buys it nothing anywhere else below.

**What linear structure it carries tracks $\Delta T$ more than $F$.** Regressing each
target on all 128 channels, the FV model gives $R^2 = 0.34 - 0.42$ for the flux against
$0.48 - 0.52$ for the bare temperature difference. The network leans on the driving
difference more than on the FV weighting that turns it into a flux, though it carries a
substantial amount of both.

**Here the FV features do buy something.** The geometric weight
$w = D_T|S_f|/(|d|\cos)$, which only the FV model's edge features encode, reads out at
$R^2 = 0.45 - 0.64$ from the FV message against $0.08 - 0.20$ from the no-FV one — and
that lead carries through to the flux itself on **every** mesh (0.34 - 0.42 vs
0.29 - 0.32) and to the node level (0.59 - 0.69 vs 0.50 - 0.56, a gain of 0.06 - 0.15).
The no-FV model compensates where it can: with no geometry to lean on, it matches or
beats the FV model on the bare $\Delta T$ on `model_003` and, narrowly, `model_007` — the
one thing left for it to encode. This is the opposite of what the same test gives on the
flange (cached in `flange_message_flux.npz`), where the FV model led only on $w$ and lost
on both $F$ and $\Delta T$.

**The aggregate is far more flux-like than any single message.** At node level, where it
actually matters, $\sum_j m_{ij}$ explains $R^2 = 0.59 - 0.69$ of the net flux
$\sum_j F_{ij}$ that drives $\mathrm{d}T_i/\mathrm{d}t$ — 1.5 to 1.9 times the per-edge
number. The network gets the cell balance broadly right while splitting it over the faces
in a way that is not, face by face, a flux. All of this holds on `model_010` and
`model_011`, whose 2-hole / 3-hole geometries postdate both checkpoints, so it is not an
artefact of the recorded test split.

The density panels show what the correlations leave out: a dense vertical spine at
$F \approx 0$, where the channel spans most of its range while the flux does not move at
all, and a cloud that widens with $|\Delta T|$ rather than following a line.

### Two things this does *not* show

- These are **linear** read-outs of a 128-dimensional message that a **nonlinear** MLP
  consumes. A low $R^2$ bounds how *directly* flux-like the message is; it does not prove
  the flux is unrecoverable from it.
- Both checkpoints are `n_mp_layers=1`, so a single message has to do all the work. A
  deeper stack has no reason to put the flux in layer 0.

### The residual model

The same read-outs for the residual checkpoint
(`parametric_history1_mesh_correct_edge_attr_residual`), on the same four held-out meshes,
the same 12 timesteps and the same seeded subset of faces as above, next to the direct FV
model's cached numbers (`FV` above, `Direct` here). Only the per-model part is recomputed,
and it is cached in `parametric_message_flux_residual.npz`. Every read-out is a
correlation, an $R^2$ or an energy fraction, so none depends on the scale the residual
network's messages take.

In [ ]:
# --- Message vs FV flux: the residual model --------------------------------
# The per-model part of compute_flux_stats above, for one checkpoint. The face
# subset and the timesteps are seeded exactly as there, so every number lines
# up row for row with the direct FV model's cached ones. Helpers (load_mesh,
# _r2, the probe and the flux) come from the flux cell above.
RESID_EXP = "parametric_history1_mesh_correct_edge_attr_residual"
flux_resid_cache = processed_data_dir / "parametric_message_flux_residual.npz"


def compute_model_flux_stats(exp):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = processed_data_dir / exp / "checkpoints"
    state = torch.load(ckpt / "model.pt", map_location=device)
    # The normalizer was fitted on the TRAINING meshes and saved with the
    # checkpoint; refitting it here would leak the held-out geometry.
    norm = FeatureNormalizer()
    norm.load(ckpt / "normalizer.pt")

    out = {}
    for mesh in FLUX_MESHES:
        graph, T_seq = load_mesh(mesh)
        case_dir = str(raw_param_dir / mesh)
        times = filter_of_time_directories(case_dir)

        ei, ea = graph.edge_index, graph.edge_attr   # RAW geometry
        N, E = graph.num_nodes, ei.shape[1]
        n_int = int((graph.node_attr[:, 0] == 1.0).sum())
        rev = paired_edge_index(ei)
        internal = (ei[0] < n_int) & (ei[1] < n_int)
        cos = ea[:, 9].double()

        # the same whole faces and timesteps as compute_flux_stats
        canon = torch.nonzero((rev > torch.arange(E)) & internal
                              & internal[rev]).squeeze(1)
        n_faces = min(KEEP_FACES, canon.numel())
        sel = canon[torch.randperm(canon.numel(),
                                   generator=torch.Generator().manual_seed(0))[:n_faces]]
        kept = torch.cat([sel, rev[sel]])
        rev_local = torch.cat([torch.arange(n_faces) + n_faces,
                               torch.arange(n_faces)])
        near = (cos[kept] > 0.99).repeat(N_STEPS)
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()

        gradT = load_cell_gradient(case_dir, [times[i] for i in steps])
        T_at = T_seq[steps].double()
        F = fv_flux_corrected(ei, ea, T_at, gradT, DT=DT_PARAM,
                              n_internal_nodes=n_int)            # (S, E)
        F_node = torch.zeros(N_STEPS, N, dtype=torch.double)
        F_node.index_add_(1, ei[1], F)                           # (S, N)
        tgts = {"F": F[:, kept].reshape(-1),
                "dT": (T_at[:, ei[0]] - T_at[:, ei[1]])[:, kept].reshape(-1),
                "w": (DT_PARAM * ea[:, 7].double()
                      / (ea[:, 3].double() * cos.clamp(min=0.05)))[kept].repeat(N_STEPS)}

        ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
        model = FVSurrogate(
            in_node_feat=HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            hidden_dim=64, out_dim=1, n_mp_layers=1,
            # the message width is read off the checkpoint's last message layer
            msg_dim=state["mp_layers.0.msg_fnc.6.weight"].shape[0],
            # residual checkpoints carry their delta_scale buffer, LayerNorm
            # ones their node_encoder_norm weights
            residual="delta_scale" in state, history=HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()

        msg_kept, aggr_all = [], []
        probe = MessageProbe(model)
        with probe, torch.no_grad():
            for i in steps:
                model(ds[i].to(device))
                msg_kept.append(probe.messages[0][kept])
                aggr_all.append(probe.aggregated[0])
                probe.clear()
        msg = torch.stack(msg_kept)                 # (S, kept, C)
        aggr = torch.stack(aggr_all)                # (S, N, C)

        # antisymmetric energy fraction of m_ij across the two halves of a face
        S_, A_ = 0.5 * (msg + msg[:, rev_local]), 0.5 * (msg - msg[:, rev_local])
        sa = A_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
        ss = S_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
        out[f"{exp}__{mesh}__anti"] = (sa / (sa + ss)).numpy()      # (C,)

        m_flat = msg.reshape(-1, msg.shape[-1]).double()
        out[f"{exp}__{mesh}__r2"] = np.array([_r2(m_flat, t) for t in tgts.values()])
        out[f"{exp}__{mesh}__r2_near"] = np.array(
            [_r2(m_flat[near], t[near]) for t in tgts.values()])
        out[f"{exp}__{mesh}__r2_node"] = _r2(
            aggr[:, :n_int].reshape(-1, msg.shape[-1]).double(),
            F_node[:, :n_int].reshape(-1))

        mc = m_flat - m_flat.mean(0, keepdim=True)
        Fc = tgts["F"] - tgts["F"].mean()
        r = (mc * Fc[:, None]).sum(0) / (mc.norm(dim=0) * Fc.norm()).clamp(min=1e-30)
        best = int(r.abs().argmax())
        # subsample for the density panels: same seed and stack size as
        # compute_flux_stats, so the same edge-samples as the direct model's
        pick = torch.randperm(m_flat.shape[0],
                              generator=torch.Generator().manual_seed(0))[:N_SCATTER]
        out[f"{exp}__{mesh}__best_ch"] = best
        out[f"{exp}__{mesh}__best_r"] = float(r[best])
        out[f"{exp}__{mesh}__sc_m"] = m_flat[pick, best].float().numpy()
        out[f"{exp}__{mesh}__sc_F"] = tgts["F"][pick].float().numpy()
        out[f"{exp}__{mesh}__sc_dT"] = tgts["dT"][pick].float().numpy()
        del msg, aggr, m_flat, probe, model
    return out


if flux_resid_cache.exists():
    flux_resid = dict(np.load(flux_resid_cache, allow_pickle=False))
else:
    flux_resid = compute_model_flux_stats(RESID_EXP)
    np.savez_compressed(flux_resid_cache, **flux_resid)

# Direct FV (cached by the flux cell above) vs residual, same columns as there.
MSG_CMP = {"Direct": FLUX_EXPS["FV"], "Residual": RESID_EXP}
flux_cmp = {**flux, **flux_resid}
print(f"{'mesh':<11}{'model':<10}{'ch':>4}{'r':>8}{'R2 F':>7}{'R2 dT':>7}{'R2 w':>7}"
      f"{'node':>7}{'anti':>7}   R2 F/dT/w on cos>0.99")
for mesh in FLUX_MESHES:
    for label, exp in MSG_CMP.items():
        print(f"{mesh:<11}{label:<10}{int(flux_cmp[f'{exp}__{mesh}__best_ch']):>4}"
              f"{float(flux_cmp[f'{exp}__{mesh}__best_r']):>+8.3f}"
              + "".join(f"{v:>7.3f}" for v in flux_cmp[f"{exp}__{mesh}__r2"])
              + f"{float(flux_cmp[f'{exp}__{mesh}__r2_node']):>7.3f}"
              f"{flux_cmp[f'{exp}__{mesh}__anti'].mean():>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux_cmp[f"{exp}__{mesh}__r2_near"]))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)

# The three edge-level read-outs of the panels above plus the node-level one,
# direct vs residual. Residual is the variant under test (PLOT1), direct the
# baseline (PLOT2), as in the direct vs residual section.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.28
models = {"Residual": (RESID_EXP, PLOT1_COLOR), "Direct": (FLUX_EXPS["FV"], PLOT2_COLOR)}

for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, (label, (exp, color)) in enumerate(models.items()):
        vals = np.append(flux_cmp[f"{exp}__{mesh}__r2"], flux_cmp[f"{exp}__{mesh}__r2_node"])
        off = (i - 0.5) * (width + 0.02)      # 2% surface gap between the pair
        bars = ax.bar(x + off, vals, width, color=color, label=label)
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=13, color=AX_COLOR)

    # R^2 stops at 1; the space above it holds the legend clear of the bars.
    legend = ax.legend(fontsize=14, loc="upper left", ncol=len(models))
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    # Mean antisymmetric energy fraction over channels (1.0 for a true flux).
    anti = ", ".join(f"{label.lower()} {flux_cmp[f'{exp}__{mesh}__anti'].mean():.2f}"
                     for label, (exp, _) in models.items())

    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 1.2)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_title(f"{mesh}\nantisymmetric energy: {anti}", fontsize=15)
    ax.set_ylabel(r"$R^2$  (all 128 channels $\rightarrow$ target)", fontsize=15)
    ax.grid(True, axis="y", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
    ax.set_axisbelow(True)

fig.suptitle(f"Message vs FV flux, direct vs residual  ({N_STEPS} probed timesteps)",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_r2_residual.png", dpi=300, transparent=True)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Sequential ramp for point density: one hue, light -> dark.
DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "fvgn_density", ["#f0eee9", "#b9a8cb", PLOT1_COLOR, "#2b1a3c"])

# The density panels above, for the residual model: its most flux-correlated
# channel against the flux and the bare temperature difference, on the same
# edge-samples as the direct model's.
fig, axs = plt.subplots(2, len(FLUX_MESHES),
                        figsize=(1600*px, 800*px), constrained_layout=True)
fig.patch.set_alpha(0)

exp = RESID_EXP
for col, mesh in enumerate(FLUX_MESHES):
    ch = int(flux_resid[f"{exp}__{mesh}__best_ch"])
    m = flux_resid[f"{exp}__{mesh}__sc_m"]
    panels = [("$F$  [K m$^3$ s$^{-1}$]", flux_resid[f"{exp}__{mesh}__sc_F"]),
              (r"$\Delta T$  [K]", flux_resid[f"{exp}__{mesh}__sc_dT"])]

    for row, (xlabel, xv) in enumerate(panels):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        hb = ax.hexbin(xv, m, gridsize=50, bins="log", cmap=DENSITY_CMAP,
                       mincnt=1, linewidths=0)
        r = np.corrcoef(xv, m)[0, 1]
        if row == 0:
            ax.set_title(f"{mesh} · channel {ch}", fontsize=17)
        ax.annotate(f"$r = {r:+.2f}$", xy=(0.03, 0.95), xycoords="axes fraction",
                    va="top", fontsize=15, color=AX_COLOR)
        ax.set_xlabel(xlabel, fontsize=15)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 3))
        ax.grid(True, ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
        ax.set_axisbelow(True)
        if col == 0:
            ax.set_ylabel("message value", fontsize=15)

cb = fig.colorbar(hb, ax=axs, pad=0.02)
cb.set_label("edge-samples per bin", fontsize=14, color=AX_COLOR)
cb.ax.tick_params(colors=AX_COLOR, labelsize=12)
cb.outline.set_edgecolor(AX_COLOR)

fig.suptitle("Most flux-correlated channel of the residual model",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_density_residual.png", dpi=300, transparent=True)

## One-channel message (`msg_dim = 1`)

The tightest possible message bottleneck: the residual model with $m_{ij}$ cut from 128
channels to **one** (`parametric_history1_msg_dim1_residual`), against the 128-channel
residual run above (`parametric_history1_mesh_correct_edge_attr_residual`). Both train on
the same six meshes for 200 epochs and are tested on the same four held-out meshes, and the
first cell below asserts that the mesh split, the normalizer and the increment scale
$s = \mathrm{rms}(\Delta T)/\sigma_T$ are identical — so the message width is what the
comparison isolates.

One channel is exactly the width of a face flux. If the network still does the job through
a scalar $m_{ij}$, the physically natural thing for that scalar to be is
$F_{j\rightarrow i}$ up to scale and sign, and the read-outs further down test exactly that.

- **Loss decay and rollout error.** `train_parametric.py` saves the rollout MAE but not the
  RMSE, so both checkpoints are rolled out again on the four held-out meshes (cached in
  `parametric_rollout_msg_dim1.npz`), and the recomputed MAE is checked against the curves
  saved at training time.
- **Message vs flux.** `compute_model_flux_stats` from the residual cell, on the same 12
  timesteps and the same seeded faces, cached in `parametric_message_flux_msg_dim1.npz`.
  With one channel every $R^2$ is a squared correlation, so the rank correlation
  $\rho_s$ is reported next to it: a message that is a monotone but *nonlinear* function
  of $F$ keeps $\rho_s$ high while $r$ drops.

In [ ]:
# --- msg_dim = 1 vs 128: loss decay and rollout errors ------------------------
# Two residual runs that differ in the message width. train_parametric.py saves
# the rollout MAE but not the RMSE, so both checkpoints are rolled out again on
# the held-out meshes, and the recomputed MAE has to reproduce the saved curves.
# Helpers (load_mesh, HISTORY, RESID_EXP) come from the flux cells above.
import json

from models.autoregressive_training import rollout

MSG1_EXP  = "parametric_history1_msg_dim1_residual"
MD_CASES  = {"msg_dim = 1": MSG1_EXP, "msg_dim = 128": RESID_EXP}
MD_MESHES = ["model_003", "model_007", "model_010", "model_011"]
md_rollout_cache = processed_data_dir / "parametric_rollout_msg_dim1.npz"

md_train_loss, md_val_loss, md_saved_mae, md_setup = {}, {}, {}, {}
for key, cname in MD_CASES.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    md_train_loss[key] = np.load(ckpt / "train_losses.npy")
    md_val_loss[key] = np.load(ckpt / "val_losses.npy")
    md_saved_mae[key] = {mesh: np.load(ckpt / f"rollout_mae_{mesh}.npy")
                         for mesh in MD_MESHES}
    with open(ckpt / "mesh_split.json") as fh:
        split = json.load(fh)
    md_setup[key] = {
        "split": split,
        "norm": torch.load(ckpt / "normalizer.pt"),
        "delta_scale": float(torch.load(ckpt / "model.pt", map_location="cpu")["delta_scale"]),
    }

# The comparison isolates the message width only if nothing else moved.
s1, s128 = md_setup["msg_dim = 1"], md_setup["msg_dim = 128"]
assert s1["split"] == s128["split"], \
    f"mesh splits differ: {s1['split']} vs {s128['split']}"
assert all(torch.equal(s1["norm"][k], s128["norm"][k]) for k in s1["norm"]), \
    "normalizers differ"
assert s1["delta_scale"] == s128["delta_scale"], \
    f"delta_scale differs: {s1['delta_scale']} vs {s128['delta_scale']}"


def compute_rollout_errors():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in MD_MESHES:
        graph, T_seq = load_mesh(mesh)
        for cname in MD_CASES.values():
            ckpt = processed_data_dir / cname / "checkpoints"
            state = torch.load(ckpt / "model.pt", map_location=device)
            norm = FeatureNormalizer()
            norm.load(ckpt / "normalizer.pt")
            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                hidden_dim=64, out_dim=1, n_mp_layers=1,
                # the message width is read off the checkpoint's last message layer
                msg_dim=state["mp_layers.0.msg_fnc.6.weight"].shape[0],
                residual="delta_scale" in state, history=HISTORY,
            ).to(device)
            model.load_state_dict(state)

            # every node, boundary included, as rollout_mae_*.npy was computed
            T_pred, T_true = rollout(model, SingleMeshDataset(T_seq, graph, norm, HISTORY),
                                     device=device)
            err = (T_pred - T_true).double()                     # (n_steps, N) [K]
            out[f"{cname}__{mesh}__mae"] = err.abs().mean(1).numpy()
            out[f"{cname}__{mesh}__rmse"] = err.pow(2).mean(1).sqrt().numpy()
            del model
    return out


if md_rollout_cache.exists():
    md_rollout = dict(np.load(md_rollout_cache, allow_pickle=False))
else:
    md_rollout = compute_rollout_errors()
    np.savez_compressed(md_rollout_cache, **md_rollout)

# If the MAE did not reproduce, the RMSE next to it would describe a different
# rollout from the one saved at training time.
mae_dev = max(float(np.abs(md_rollout[f"{cname}__{mesh}__mae"] - md_saved_mae[key][mesh]).max())
              for key, cname in MD_CASES.items() for mesh in MD_MESHES)
print(f"recomputed vs saved rollout MAE: max |difference| = {mae_dev:.1e} K\n")

print(f"{'':18s}" + "".join(f"{key:>17s}" for key in MD_CASES))
print(f"{'train loss, last':18s}" + "".join(f"{md_train_loss[k][-1]:>17.3e}" for k in MD_CASES))
print(f"{'val loss, last':18s}" + "".join(f"{md_val_loss[k][-1]:>17.3e}" for k in MD_CASES))
print(f"{'val loss, best':18s}" + "".join(
    f"{f'{md_val_loss[k].min():.3e} @{md_val_loss[k].argmin()}':>17s}" for k in MD_CASES))

c1, c128 = MD_CASES["msg_dim = 1"], MD_CASES["msg_dim = 128"]
print(f"\n{'':10s}{'rollout MAE [K]':^36s}{'rollout RMSE [K]':^36s}")
print(f"{'':10s}" + f"{'mean':^18s}{'last step':^18s}" * 2)
print(f"{'msg_dim':10s}" + f"{'1':>9s}{'128':>9s}" * 4)
for mesh in MD_MESHES:
    row = f"{mesh:10s}"
    for metric in ("mae", "rmse"):
        e1, e128 = md_rollout[f"{c1}__{mesh}__{metric}"], md_rollout[f"{c128}__{mesh}__{metric}"]
        row += f"{e1.mean():9.3f}{e128.mean():9.3f}{e1[-1]:9.3f}{e128[-1]:9.3f}"
    print(row)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# msg_dim = 1 is the variant under test (solid, PLOT1) and the 128-channel
# residual run the baseline (dashed, PLOT2), the roles residual / direct take
# above. Shared y-axis, so the training and validation panels read on one scale.
md_styles = {"msg_dim = 1": (PLOT1_COLOR, "-"), "msg_dim = 128": (PLOT2_COLOR, "--")}

fig, axs = plt.subplots(1, 2, figsize=(1400*px, 540*px), sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Loss Decay, msg_dim = 1 vs 128, Residual FV, h = 1",
             fontsize=24, color=AX_COLOR)

for ax, (title, losses) in zip(axs, (("Training Loss", md_train_loss),
                                     ("Validation Loss", md_val_loss))):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, (color, ls) in md_styles.items():
        ax.plot(losses[key], label=key, color=color, linewidth=2.5, linestyle=ls)

    # Value where each curve ends: the higher one labelled above its line and
    # the lower one below, so the two labels cannot collide.
    ends = sorted(((losses[k][-1], k) for k in md_styles), reverse=True)
    for (y, key), dy, va in zip(ends, (8, -8), ("bottom", "top")):
        ax.annotate(f"{y:.1e}", xy=(len(losses[key]) - 1, y), xytext=(0, dy),
                    textcoords="offset points", ha="right", va=va,
                    fontsize=14, color=AX_COLOR)

    ax.set_yscale("log")

    legend = ax.legend(fontsize=18)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if ax is axs[0]:
        ax.set_ylabel("MSE Loss", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_msg_dim1_loss.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

md_styles = {"msg_dim = 1": (PLOT1_COLOR, "-"), "msg_dim = 128": (PLOT2_COLOR, "--")}

# Rows: rollout MAE and RMSE; columns: the held-out meshes, each run at its
# final checkpoint. RMSE >= MAE at every step, and the gap between the two is
# how concentrated the error is on a few cells.
fig, axs = plt.subplots(2, len(MD_MESHES), figsize=(1600*px, 800*px),
                        sharex=True, constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Rollout Error Accumulation, msg_dim = 1 vs 128, h = 1",
             fontsize=24, color=AX_COLOR)

for row, (metric, ylabel) in enumerate((("mae", "MAE [K]"), ("rmse", "RMSE [K]"))):
    for col, mesh in enumerate(MD_MESHES):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key, (color, ls) in md_styles.items():
            ax.plot(md_rollout[f"{MD_CASES[key]}__{mesh}__{metric}"], label=key,
                    color=color, linewidth=2.5, linestyle=ls)
        ax.set_ylim(bottom=0)

        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        if col == 0:
            ax.set_ylabel(ylabel, fontsize=15)
        ax.tick_params(axis='both', labelsize=13)
        ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

legend = axs[0, 0].legend(fontsize=14, loc="upper left")
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

# plt.savefig("../outputs/plots/parametric_history1_msg_dim1_rollout.png", dpi=300, transparent = True)

In [ ]:
# --- Message vs FV flux: the one-channel message -------------------------------
# compute_model_flux_stats from the residual cell, on the same meshes, timesteps
# and seeded faces, so every row lines up with the 128-channel residual model's.
# With one channel there is nothing to pick (the best channel is channel 0) and
# every R^2 is a squared correlation, so the Spearman rank correlation with F is
# printed next to r: a monotone but nonlinear m = f(F) keeps rho high while r
# drops. rho is taken on the density-panel subsample; for the 128-channel model
# both r and rho describe its most flux-correlated channel.
from scipy.stats import spearmanr

flux_md1_cache = processed_data_dir / "parametric_message_flux_msg_dim1.npz"

if flux_md1_cache.exists():
    flux_md1 = dict(np.load(flux_md1_cache, allow_pickle=False))
else:
    flux_md1 = compute_model_flux_stats(MSG1_EXP)
    np.savez_compressed(flux_md1_cache, **flux_md1)

MD_MSG = {"msg_dim = 128": RESID_EXP, "msg_dim = 1": MSG1_EXP}
flux_md = {**flux_resid, **flux_md1}
print(f"{'mesh':<11}{'model':<15}{'ch':>4}{'r':>8}{'rho':>8}{'R2 F':>7}{'R2 dT':>7}{'R2 w':>7}"
      f"{'node':>7}{'anti':>7}   R2 F/dT/w on cos>0.99")
for mesh in FLUX_MESHES:
    for label, exp in MD_MSG.items():
        rho = spearmanr(flux_md[f"{exp}__{mesh}__sc_m"], flux_md[f"{exp}__{mesh}__sc_F"])[0]
        print(f"{mesh:<11}{label:<15}{int(flux_md[f'{exp}__{mesh}__best_ch']):>4}"
              f"{float(flux_md[f'{exp}__{mesh}__best_r']):>+8.3f}{rho:>+8.3f}"
              + "".join(f"{v:>7.3f}" for v in flux_md[f"{exp}__{mesh}__r2"])
              + f"{float(flux_md[f'{exp}__{mesh}__r2_node']):>7.3f}"
              f"{flux_md[f'{exp}__{mesh}__anti'].mean():>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux_md[f"{exp}__{mesh}__r2_near"]))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)

# The three edge-level read-outs plus the node-level one, one channel vs 128.
# For msg_dim = 1 each bar is a squared correlation; for 128 it is the best
# linear combination of all channels, so the wide message has the easier test.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.28
models = {"msg_dim = 1": (MSG1_EXP, PLOT1_COLOR), "msg_dim = 128": (RESID_EXP, PLOT2_COLOR)}

for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, (label, (exp, color)) in enumerate(models.items()):
        vals = np.append(flux_md[f"{exp}__{mesh}__r2"], flux_md[f"{exp}__{mesh}__r2_node"])
        off = (i - 0.5) * (width + 0.02)      # 2% surface gap between the pair
        bars = ax.bar(x + off, vals, width, color=color, label=label)
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=13, color=AX_COLOR)

    # R^2 stops at 1; the space above it holds the legend clear of the bars.
    legend = ax.legend(fontsize=14, loc="upper left", ncol=len(models))
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    # Mean antisymmetric energy fraction over channels (1.0 for a true flux).
    anti = ", ".join(f"{flux_md[f'{exp}__{mesh}__anti'].mean():.2f} ({label.split()[-1]} ch)"
                     for label, (exp, _) in models.items())

    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 1.2)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_title(f"{mesh}\nantisymmetric energy: {anti}", fontsize=15)
    ax.set_ylabel(r"$R^2$  (all channels $\rightarrow$ target)", fontsize=15)
    ax.grid(True, axis="y", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
    ax.set_axisbelow(True)

fig.suptitle(f"Message vs FV flux, msg_dim = 1 vs 128  ({N_STEPS} probed timesteps)",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_r2_msg_dim1.png", dpi=300, transparent=True)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Sequential ramp for point density: one hue, light -> dark.
DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "fvgn_density", ["#f0eee9", "#b9a8cb", PLOT1_COLOR, "#2b1a3c"])

# The density panels above, for the one-channel model. No channel is selected
# here: this is the whole message, on the same edge-samples as the direct and
# residual models' panels.
fig, axs = plt.subplots(2, len(FLUX_MESHES),
                        figsize=(1600*px, 800*px), constrained_layout=True)
fig.patch.set_alpha(0)

exp = MSG1_EXP
for col, mesh in enumerate(FLUX_MESHES):
    m = flux_md1[f"{exp}__{mesh}__sc_m"]
    panels = [("$F$  [K m$^3$ s$^{-1}$]", flux_md1[f"{exp}__{mesh}__sc_F"]),
              (r"$\Delta T$  [K]", flux_md1[f"{exp}__{mesh}__sc_dT"])]

    for row, (xlabel, xv) in enumerate(panels):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        hb = ax.hexbin(xv, m, gridsize=50, bins="log", cmap=DENSITY_CMAP,
                       mincnt=1, linewidths=0)
        r = np.corrcoef(xv, m)[0, 1]
        rho = spearmanr(xv, m)[0]
        if row == 0:
            ax.set_title(mesh, fontsize=17)
        ax.annotate(f"$r = {r:+.2f}$\n$\\rho_s = {rho:+.2f}$", xy=(0.03, 0.95),
                    xycoords="axes fraction", va="top", fontsize=15, color=AX_COLOR)
        ax.set_xlabel(xlabel, fontsize=15)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 3))
        ax.grid(True, ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
        ax.set_axisbelow(True)
        if col == 0:
            ax.set_ylabel("message $m_{ij}$", fontsize=15)

cb = fig.colorbar(hb, ax=axs, pad=0.02)
cb.set_label("edge-samples per bin", fontsize=14, color=AX_COLOR)
cb.ax.tick_params(colors=AX_COLOR, labelsize=12)
cb.outline.set_edgecolor(AX_COLOR)

fig.suptitle("The one-channel message against the flux",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_density_msg_dim1.png", dpi=300, transparent=True)

In [ ]:
# --- Is the one-channel message used, and what does it carry? -----------------
# A message this unlike the flux could simply be ignored by node_fnc. Two
# ablations on the same 12 timesteps answer that, for both widths: replace every
# m_ij by the message's mean over those steps (no information left), or shuffle
# the messages across edges (right distribution, wrong faces). The one-step MSE
# is in normalized T, the training loss's units, next to the persistence
# baseline T^{n+1} = T^n. Alongside, what the one-channel message tracks
# linearly instead: the temperatures on the two sides of its face, their sum and
# their difference, on the same seeded faces as the flux read-outs.
msg_usage_cache = processed_data_dir / "parametric_message_usage_msg_dim1.npz"


def compute_message_usage():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in FLUX_MESHES:
        graph, T_seq = load_mesh(mesh)
        ei = graph.edge_index
        E = ei.shape[1]
        n_int = int((graph.node_attr[:, 0] == 1.0).sum())
        rev = paired_edge_index(ei)
        internal = (ei[0] < n_int) & (ei[1] < n_int)

        # the same whole faces and timesteps as compute_flux_stats
        canon = torch.nonzero((rev > torch.arange(E)) & internal
                              & internal[rev]).squeeze(1)
        n_faces = min(KEEP_FACES, canon.numel())
        sel = canon[torch.randperm(canon.numel(),
                                   generator=torch.Generator().manual_seed(0))[:n_faces]]
        kept = torch.cat([sel, rev[sel]])
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()

        for exp in MD_CASES.values():
            ckpt = processed_data_dir / exp / "checkpoints"
            state = torch.load(ckpt / "model.pt", map_location=device)
            norm = FeatureNormalizer()
            norm.load(ckpt / "normalizer.pt")
            width = state["mp_layers.0.msg_fnc.6.weight"].shape[0]
            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                hidden_dim=64, out_dim=1, n_mp_layers=1, msg_dim=width,
                residual="delta_scale" in state, history=HISTORY,
            ).to(device)
            model.load_state_dict(state)
            model.eval()
            ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
            batches = [ds[i].to(device) for i in steps]

            # true messages: their mean feeds the ablation, and for the one-channel
            # model the kept faces feed the read-outs below
            mse = {"true": [], "mean": [], "shuffle": []}
            msg_sum, msg_n, msg_kept = 0.0, 0, []
            probe = MessageProbe(model)
            with probe, torch.no_grad():
                for d in batches:
                    mse["true"].append(float(((model(d) - d.y) ** 2).mean()))
                    m = probe.messages[0]
                    msg_sum, msg_n = msg_sum + m.double().sum(0), msg_n + m.shape[0]
                    if width == 1:
                        msg_kept.append(m[kept, 0])
                    probe.clear()

            # a message forward hook that returns a tensor replaces the message
            msg_mean = (msg_sum / msg_n).float().to(device)
            gen = torch.Generator().manual_seed(0)
            replace = {
                "mean": lambda out: msg_mean.expand_as(out),
                "shuffle": lambda out: out[torch.randperm(out.shape[0], generator=gen)
                                           .to(out.device)],
            }
            for mode, fn in replace.items():
                handle = model.mp_layers[0].register_message_forward_hook(
                    lambda module, inputs, out, fn=fn: fn(out))
                with torch.no_grad():
                    for d in batches:
                        mse[mode].append(float(((model(d) - d.y) ** 2).mean()))
                handle.remove()
            for mode, vals in mse.items():
                out[f"{exp}__{mesh}__mse_{mode}"] = float(np.mean(vals))
            out[f"{mesh}__mse_persistence"] = float(np.mean(
                [float(((d.x[:, HISTORY - 1] - d.y) ** 2).mean()) for d in batches]))

            if width == 1:
                m_flat = torch.stack(msg_kept).reshape(-1, 1).double()
                T_at = T_seq[steps].double()
                Ti, Tj = T_at[:, ei[1]][:, kept], T_at[:, ei[0]][:, kept]   # receiver, sender
                out[f"{exp}__{mesh}__r2_levels"] = np.array(
                    [_r2(m_flat, t.reshape(-1)) for t in (Ti, Tj, Ti + Tj, Tj - Ti)])
            del model, batches
    return out


if msg_usage_cache.exists():
    msg_usage = dict(np.load(msg_usage_cache, allow_pickle=False))
else:
    msg_usage = compute_message_usage()
    np.savez_compressed(msg_usage_cache, **msg_usage)

print(f"one-step MSE (normalized T) on the {N_STEPS} probed timesteps")
print(f"{'mesh':<11}{'model':<15}{'true m':>10}{'mean m':>10}{'shuffled m':>12}{'persistence':>13}")
for mesh in FLUX_MESHES:
    for label, exp in MD_CASES.items():
        print(f"{mesh:<11}{label:<15}"
              + "".join(f"{msg_usage[f'{exp}__{mesh}__mse_{mode}']:>{w}.2e}"
                        for mode, w in (("true", 10), ("mean", 10), ("shuffle", 12)))
              + f"{msg_usage[f'{mesh}__mse_persistence']:>13.2e}")

print(f"\nR^2 of the one-channel message on")
print(f"{'mesh':<11}{'T_i':>7}{'T_j':>7}{'T_i+T_j':>9}{'T_j-T_i':>9}")
for mesh in FLUX_MESHES:
    print(f"{mesh:<11}" + "".join(f"{v:>{w}.3f}" for v, w in
                                  zip(msg_usage[f"{MSG1_EXP}__{mesh}__r2_levels"], (7, 7, 9, 9))))

### What comes out

**Loss.** One channel trains to a validation MSE of $5.0\times10^{-5}$, against
$2.8\times10^{-5}$ with 128 (best $4.4\times10^{-5}$ at epoch 60 vs $2.5\times10^{-5}$ at
epoch 79): about 1.8 times higher, with both runs levelling off after epoch ~90. The
bottleneck costs one-step accuracy, but not an order of magnitude of it.

**Rollout.** That gap does not carry through uniformly. On `model_003` and `model_007` the
one-channel model accumulates *less* error than the 128-channel one — mean MAE 5.98 vs
6.06 K and 3.12 vs 5.23 K, last-step RMSE 9.7 vs 16.4 K and 8.0 vs 16.7 K — flattening out
where the wide model keeps climbing. On the 2-hole / 3-hole geometries `model_010` and
`model_011` it is worse: mean MAE 6.84 vs 6.19 K and 6.68 vs 5.06 K, last-step RMSE 17.9
vs 13.3 K and 12.1 vs 9.4 K. For both models the RMSE runs 1.4 – 2.3 times the MAE, so the
error sits on a minority of cells. Each width is a single training run, so the per-mesh
ordering cannot be separated from seed-to-seed spread here.

**The message is not the flux.** With a single channel every read-out is one squared
correlation, and none of them moves: $R^2 = 0.02 - 0.04$ for $F$, $0.02 - 0.05$ for
$\Delta T$, $0.00 - 0.09$ for $w$, and $0.00 - 0.01$ between $\sum_j m_{ij}$ and the net
flux. The 128-channel model's full linear read-out reaches $0.76 - 0.82$ for $F$, and even
its single most flux-correlated channel ($r = +0.23 \ldots +0.39$) beats the one-channel
message ($r = -0.15 \ldots -0.20$). The rank correlation is no better
($\rho_s = -0.06 \ldots -0.14$), so no monotone nonlinear relation is hiding behind the low
$r$ either. The antisymmetric energy fraction is **0.04** on every mesh: the message is
almost exactly *symmetric* across a face, $m_{ij} \approx m_{ji}$ — the opposite of a
conservative flux, and further from one than the 128-channel message (0.25 – 0.28).

**But the network does use it.** Replacing every $m_{ij}$ by the message's mean raises the
one-step MSE 7 – 22 fold, to just below the persistence baseline $T^{n+1} = T^n$, and
shuffling the messages across edges is worse than persistence. The 128-channel model
behaves the same way. The one channel carries essentially all the neighbour information the
update receives; it is simply not organised as a flux.

**What it tracks instead is the temperature level.** The message follows the temperatures
on the two sides of its face — $R^2 = 0.16 - 0.55$ on $T_i$, $0.18 - 0.57$ on $T_j$,
$0.17 - 0.56$ on $T_i + T_j$ — against $0.02 - 0.05$ on their difference. A symmetric,
level-like message is enough because `node_fnc` is nonlinear and also receives $T_i$: the
differencing a flux does face by face can instead happen after aggregation, against the
cell's own temperature. The flux is one solution to the one-channel problem, not the only
one, and the width constraint alone does not select it.

**What would select it.** Build the antisymmetry in,
$m_{ij} = \tfrac12\left[f(x_i, x_j, e_{ij}) - f(x_j, x_i, e_{ji})\right]$. A level-like
message then cancels across every face by construction, and $\sum_j m_{ij}$ is conservative.

## PCA spectrum of the message

`msg_dim = 1` costs a factor ~1.8 in validation loss, so one channel is too few, but that
does not say how many channels the 128-channel message actually uses. Its PCA spectrum
does, straight from the trained checkpoint: the eigenvalues
$\lambda_1 \ge \dots \ge \lambda_{128}$ of the channel covariance of the `msg_dim = 128`
residual model (`parametric_history1_mesh_correct_edge_attr_residual`), on the same four
held-out meshes and 12 timesteps as above. Unlike the flux read-outs nothing is subsampled:
the covariance is accumulated over every edge and every node, and cached in
`parametric_message_pca_residual.npz`.

The spectrum does not care which channels carry the message. `node_fnc` opens with a
`Linear` layer, which absorbs any rotation of its input, so a message confined to a
$k$-dimensional subspace is a $k$-channel message in whatever basis it is written.

Two levels:

- **edge**, $m_{ij}$ — what a width bottleneck on `msg_dim` constrains;
- **node**, $\sum_j m_{ij}$ — what `node_fnc` actually receives. A direction that sums to
  zero around every cell adds variance to the edge spectrum but not to the node one.

Each spectrum is summarised by its participation ratio
$d_{PR} = \left(\sum_k \lambda_k\right)^2 / \sum_k \lambda_k^2$ — exactly $k$ for $k$ equal
eigenvalues and zeros elsewhere — and by the number of components that reaches 90, 99 and
99.9 % of the variance.

In [ ]:
# --- PCA spectrum of the 128-channel message ------------------------------------
# Channel covariance of m_ij over EVERY edge and of sum_j m_ij over every node of
# a held-out mesh, pooled over the same N_STEPS timesteps as the flux read-outs.
# Only running first and second moments are kept, never the (steps, E, 128)
# stack, so unlike the read-outs above nothing is subsampled. The cache holds the
# covariances, not just their eigenvalues, so other projections of the message
# need no new pass. Helpers (load_mesh, HISTORY, RESID_EXP, the probe) come from
# the flux cells above.
pca_cache = processed_data_dir / "parametric_message_pca_residual.npz"
PCA_THRESH = (0.90, 0.99, 0.999)


def compute_message_covariance(exp):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = processed_data_dir / exp / "checkpoints"
    state = torch.load(ckpt / "model.pt", map_location=device)
    # The normalizer was fitted on the TRAINING meshes and saved with the
    # checkpoint; refitting it here would leak the held-out geometry.
    norm = FeatureNormalizer()
    norm.load(ckpt / "normalizer.pt")

    out = {}
    for mesh in FLUX_MESHES:
        graph, T_seq = load_mesh(mesh)
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()
        ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
        model = FVSurrogate(
            in_node_feat=HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            hidden_dim=64, out_dim=1, n_mp_layers=1,
            msg_dim=state["mp_layers.0.msg_fnc.6.weight"].shape[0],
            residual="delta_scale" in state, history=HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()

        moments = {"edge": [0, 0, 0], "node": [0, 0, 0]}   # sum x, sum x x^T, count
        probe = MessageProbe(model, to_cpu=False)
        with probe, torch.no_grad():
            for i in steps:
                model(ds[i].to(device))
                for level, x in (("edge", probe.messages[0]), ("node", probe.aggregated[0])):
                    x = x.double()
                    mom = moments[level]
                    mom[0], mom[1], mom[2] = mom[0] + x.sum(0), mom[1] + x.T @ x, mom[2] + x.shape[0]
                probe.clear()

        for level, (s, ss, n) in moments.items():
            mean = s / n
            out[f"{exp}__{mesh}__{level}_cov"] = (ss / n - torch.outer(mean, mean)).cpu().numpy()
        del probe, model
    return out


def message_spectrum(exp, mesh, level):
    """Eigenvalues of the cached channel covariance, largest first."""
    return np.linalg.eigvalsh(msg_pca[f"{exp}__{mesh}__{level}_cov"])[::-1].clip(min=0)


def spectrum_summary(lam):
    """Participation ratio (sum lam)^2 / sum lam^2, and the number of components
    that reaches each fraction of the variance in PCA_THRESH."""
    frac = np.cumsum(lam) / lam.sum()
    return (lam.sum() ** 2 / (lam ** 2).sum(),
            [int(np.searchsorted(frac, t)) + 1 for t in PCA_THRESH])


if pca_cache.exists():
    msg_pca = dict(np.load(pca_cache, allow_pickle=False))
else:
    msg_pca = compute_message_covariance(RESID_EXP)
    np.savez_compressed(pca_cache, **msg_pca)

print(f"PCA of the msg_dim = 128 residual message, {N_STEPS} probed timesteps")
print(f"{'':25s}{'variance share':^21s}{'components for':^24s}")
print(f"{'mesh':<11}{'level':<7}{'d_PR':>7}{'PC1':>7}{'PC1-2':>7}{'PC1-3':>7}"
      + "".join(f"{f'{100*t:g}%':>8}" for t in PCA_THRESH))
for mesh in FLUX_MESHES:
    for level in ("edge", "node"):
        lam = message_spectrum(RESID_EXP, mesh, level)
        d_pr, n_comp = spectrum_summary(lam)
        share = np.cumsum(lam[:3]) / lam.sum()
        print(f"{mesh:<11}{level:<7}{d_pr:>7.2f}" + "".join(f"{v:>7.2f}" for v in share)
              + "".join(f"{k:>8d}" for k in n_comp))

In [ ]:
from matplotlib.ticker import NullLocator

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The edge message m_ij is what a width bottleneck constrains (solid, PLOT1) and
# its sum at the receiver what node_fnc actually receives (dashed, PLOT2).
# Rows: the spectrum itself, and the variance the first k components leave
# unexplained, on which 90 / 99 / 99.9 % are the 1e-1 / 1e-2 / 1e-3 lines.
pca_styles = {"edge": (r"edge  $m_{ij}$", PLOT1_COLOR, "-"),
              "node": (r"node  $\Sigma_j\, m_{ij}$", PLOT2_COLOR, "--")}
K_TICKS = [1, 2, 4, 8, 16, 32, 64, 128]
# The last few eigenvalues fall to round-off (~1e-15 of the total); showing them
# would spend most of each axis on digits that carry nothing.
Y_FLOOR = 1e-7

fig, axs = plt.subplots(2, len(FLUX_MESHES), figsize=(1600*px, 800*px),
                        sharex=True, sharey="row", constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("PCA Spectrum of the Message, msg_dim = 128, Residual FV, h = 1",
             fontsize=24, color=AX_COLOR)

for col, mesh in enumerate(FLUX_MESHES):
    for ax in axs[:, col]:
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)
        # ax.set_yscale("log")
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

    d_pr = []
    for level, (label, color, ls) in pca_styles.items():
        lam = message_spectrum(RESID_EXP, mesh, level)
        k = np.arange(1, lam.size + 1)
        d_pr.append(f"{level} {spectrum_summary(lam)[0]:.1f}")
        axs[0, col].plot(k, lam / lam.sum(), label=label,
                         color=color, linewidth=2.5, linestyle=ls)
        axs[1, col].plot(k, 1 - np.cumsum(lam) / lam.sum(), label=label,
                         color=color, linewidth=2.5, linestyle=ls)

    for t in PCA_THRESH:
        axs[1, col].axhline(1 - t, color=AX_COLOR, linewidth=1.5, linestyle=":")
        # labelled below the line at the left, where the curves are still above all three
        axs[1, col].annotate(f"{100*t:g} %", xy=(0.0, 1 - t), xycoords=("axes fraction", "data"),
                             xytext=(4, -3), textcoords="offset points", ha="left", va="top",
                             fontsize=12, color=AX_COLOR)

    axs[0, col].set_title(mesh, fontsize=19)
    axs[0, col].annotate(r"$d_{PR}$: " + ", ".join(d_pr), xy=(0.97, 0.95),
                         xycoords="axes fraction", ha="right", va="top",
                         fontsize=14, color=AX_COLOR)
    axs[1, col].set_xlabel("Principal component $k$", fontsize=15)

# shared axes: setting the scale and ticks once applies to every panel
axs[1, 0].set_xscale("log", base=2)
axs[1, 0].set_xticks(K_TICKS, labels=[str(k) for k in K_TICKS])
axs[1, 0].xaxis.set_minor_locator(NullLocator())
axs[1, 0].set_xlim(1, K_TICKS[-1])
for ax in axs[:, 0]:
    ax.set_ylim(Y_FLOOR, 1.5)

axs[0, 0].set_ylabel(r"$\lambda_k \,/\, \Sigma\, \lambda$", fontsize=15)
axs[1, 0].set_ylabel(r"$1 - \Sigma_{i \leq k}\, \lambda_i \,/\, \Sigma\, \lambda$", fontsize=15)

legend = axs[0, 0].legend(fontsize=14, loc="lower left")
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

# plt.savefig("../outputs/plots/parametric_message_pca_residual.png", dpi=300, transparent=True)

### What comes out

**No clean cutoff per edge.** The spectrum of $m_{ij}$ decays smoothly, with no gap after a
handful of components that would name a width. The first component carries only 26 – 36 %
of the variance and the participation ratio is $d_{PR} = 6.0 - 8.2$; 90 % of the variance
takes 11 – 14 components, 99 % takes 37 – 42 and 99.9 % takes 66 – 70. The last four
eigenvalues are zero to round-off (below $10^{-12}$ of the total, off the bottom of the axes),
so the message spans at most 124 of its 128 dimensions, and another 11 – 12 sit below
$10^{-7}$.

**The aggregate is far more concentrated.** At node level $d_{PR} = 2.1 - 2.9$: the first
component of $\sum_j m_{ij}$ carries 54 – 67 % of its variance and the first three 82 – 87 %,
and 90 % takes 5 – 7 components against 11 – 14 per edge. From $k = 3$ down to the round-off
tail the node spectrum lies below the edge one on every mesh, so summing over a cell's faces
piles the variance onto a few leading directions. Not all of that is temperature
information: every node receives its number of faces times the mean message, which puts
variance along the mean direction wherever the face count changes from node to node.

**What this says about the width.** The spectrum counts the directions the message spreads
its variance over, not the ones `node_fnc` needs: a low-variance direction still matters if
the first `Linear` of `node_fnc` weighs it heavily, and a high-variance one can be ignored.
So it gives no width to hard-code, only a range — one channel is below even the node-level
$d_{PR}$, about ten keep 90 % of the edge variance, and past ~40 only a 1 % tail is left.
Whether that tail matters is a question for the loss (a sweep over `msg_dim`, or a
bottleneck that lets training choose), not for the spectrum.

## LayerNorm (`layer_norm = True`)

The 128-channel residual model with LayerNorm added
(`parametric_history1_msg_dim128_layernorm_residual`), against the same model without it
(`parametric_history1_mesh_correct_edge_attr_residual`, the `msg_dim = 128` run above). A
LayerNorm follows the node and edge encoders and every message $m_{ij}$, before the sum at the
receiver; the node MLP of the single MP layer decodes $T$ and is left plain. Both runs train on
the same six meshes for 200 epochs, and the first cell below asserts that the mesh split, the
normalizer and the increment scale $s$ are identical, so LayerNorm is what the comparison
isolates.

- **Loss decay.** Training and validation MSE as saved by `train_parametric.py`.
- **RMAE and RMSE** exactly as `test_parametric.py` prints them, in kelvin, summed over every
  node and sample of a held-out mesh:
  $\mathrm{RMAE} = \sum|T_{pred}-T_{true}| \,/\, \sum|T_{true}|$ and
  $\mathrm{RMSE} = \sum(T_{pred}-T_{true})^2 \,/\, \sum T_{true}^2$ — a relative *squared*
  error, with no square root. Reported for the one-step predictions, as there, and for the
  autoregressive rollout, whose MAE is checked against the curves saved at training time.
  Cached in `parametric_errors_layernorm.npz`.
- **Message vs flux, $R^2$ only.** `compute_model_flux_stats` from the residual cell, on the
  same 12 timesteps and seeded faces, cached in `parametric_message_flux_layernorm.npz`. The
  probe records the message *after* the LayerNorm, since that is what gets summed: before
  LayerNorm's learned scale and shift, every $m_{ij}$ has zero mean and unit spread over its
  128 channels, so the flux can only be carried by the message's direction, not its size.

In [ ]:
# --- LayerNorm vs plain, 128-channel residual: loss, RMAE and RMSE -------------
# RMAE and RMSE as test_parametric.py prints them: sums over every node and every
# sample of a held-out mesh, in kelvin, where "RMSE" is the relative SQUARED error
# (no square root). Computed for the one-step predictions, as there, and over the
# rollout. Helpers (load_mesh, HISTORY, RESID_EXP) come from the flux cells above.
import json

from models.autoregressive_training import rollout

LN_EXP    = "parametric_history1_msg_dim128_layernorm_residual"
LN_CASES  = {"no LayerNorm": RESID_EXP, "LayerNorm": LN_EXP}
LN_MESHES = ["model_003", "model_007", "model_010", "model_011"]
ln_errors_cache = processed_data_dir / "parametric_errors_layernorm.npz"

ln_train_loss, ln_val_loss, ln_saved_mae, ln_setup = {}, {}, {}, {}
for key, cname in LN_CASES.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    ln_train_loss[key] = np.load(ckpt / "train_losses.npy")
    ln_val_loss[key] = np.load(ckpt / "val_losses.npy")
    ln_saved_mae[key] = {mesh: np.load(ckpt / f"rollout_mae_{mesh}.npy")
                         for mesh in LN_MESHES}
    with open(ckpt / "mesh_split.json") as fh:
        split = json.load(fh)
    state = torch.load(ckpt / "model.pt", map_location="cpu")
    ln_setup[key] = {
        "split": split,
        "norm": torch.load(ckpt / "normalizer.pt"),
        "delta_scale": float(state["delta_scale"]),
        "msg_dim": state["mp_layers.0.msg_fnc.6.weight"].shape[0],
        "layer_norm": "node_encoder_norm.weight" in state,
    }

# The comparison isolates LayerNorm only if nothing else moved.
s0, s1 = ln_setup["no LayerNorm"], ln_setup["LayerNorm"]
assert s0["split"] == s1["split"], \
    f"mesh splits differ: {s0['split']} vs {s1['split']}"
assert all(torch.equal(s0["norm"][k], s1["norm"][k]) for k in s0["norm"]), \
    "normalizers differ"
assert s0["delta_scale"] == s1["delta_scale"], \
    f"delta_scale differs: {s0['delta_scale']} vs {s1['delta_scale']}"
assert s0["msg_dim"] == s1["msg_dim"] == 128, \
    f"msg_dim differs: {s0['msg_dim']} vs {s1['msg_dim']}"
assert (s0["layer_norm"], s1["layer_norm"]) == (False, True), \
    "the two checkpoints are not LayerNorm off / on"


def compute_ln_errors():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in LN_MESHES:
        graph, T_seq = load_mesh(mesh)
        for cname in LN_CASES.values():
            ckpt = processed_data_dir / cname / "checkpoints"
            state = torch.load(ckpt / "model.pt", map_location=device)
            norm = FeatureNormalizer()
            norm.load(ckpt / "normalizer.pt")
            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                hidden_dim=64, out_dim=1, n_mp_layers=1,
                msg_dim=state["mp_layers.0.msg_fnc.6.weight"].shape[0],
                residual="delta_scale" in state, history=HISTORY,
                layer_norm="node_encoder_norm.weight" in state,
            ).to(device)
            model.load_state_dict(state)
            model.eval()
            ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)

            # one-step: every sample sees the true T^n. The four sums are
            # accumulated as it goes instead of stacking (n_samples, N) twice.
            sums = torch.zeros(4, dtype=torch.float64)   # |e|, |T|, e^2, T^2
            with torch.no_grad():
                for i in range(len(ds)):
                    d = ds[i].to(device)
                    T_p = norm.inverse_transform_T(model(d).cpu()).double()
                    T_t = norm.inverse_transform_T(d.y.cpu()).double()
                    e = T_p - T_t
                    sums += torch.stack([e.abs().sum(), T_t.abs().sum(),
                                         e.pow(2).sum(), T_t.pow(2).sum()])
            out[f"{cname}__{mesh}__rmae_step"] = float(sums[0] / sums[1])
            out[f"{cname}__{mesh}__rmse_step"] = float(sums[2] / sums[3])

            T_pred, T_true = rollout(model, ds, device=device)
            e, T_true = (T_pred - T_true).double(), T_true.double()   # (n_steps, N) [K]
            out[f"{cname}__{mesh}__rmae_roll"] = float(e.abs().sum() / T_true.abs().sum())
            out[f"{cname}__{mesh}__rmse_roll"] = float(e.pow(2).sum() / T_true.pow(2).sum())
            out[f"{cname}__{mesh}__mae_roll"] = e.abs().mean(1).numpy()
            del model
    return out


if ln_errors_cache.exists():
    ln_errors = dict(np.load(ln_errors_cache, allow_pickle=False))
else:
    ln_errors = compute_ln_errors()
    np.savez_compressed(ln_errors_cache, **ln_errors)

# If the rollout MAE did not reproduce, the rollout RMAE / RMSE would describe a
# different rollout from the one saved at training time.
mae_dev = max(float(np.abs(ln_errors[f"{cname}__{mesh}__mae_roll"] - ln_saved_mae[key][mesh]).max())
              for key, cname in LN_CASES.items() for mesh in LN_MESHES)
print(f"recomputed vs saved rollout MAE: max |difference| = {mae_dev:.1e} K\n")

print(f"{'':18s}" + "".join(f"{key:>17s}" for key in LN_CASES))
print(f"{'train loss, last':18s}" + "".join(f"{ln_train_loss[k][-1]:>17.3e}" for k in LN_CASES))
print(f"{'val loss, last':18s}" + "".join(f"{ln_val_loss[k][-1]:>17.3e}" for k in LN_CASES))
print(f"{'val loss, best':18s}" + "".join(
    f"{f'{ln_val_loss[k].min():.3e} @{ln_val_loss[k].argmin()}':>17s}" for k in LN_CASES))

# Columns: (one-step, rollout) x (RMAE, RMSE), each without / with LayerNorm.
cols = [(mode, metric) for mode in ("step", "roll") for metric in ("rmae", "rmse")]
pair = (LN_CASES["no LayerNorm"], LN_CASES["LayerNorm"])
print(f"\n{'':10s}{'one-step':^44s}{'rollout':^44s}")
print(f"{'':10s}" + "".join(f"{metric.upper():^22s}" for _, metric in cols))
print(f"{'mesh':10s}" + f"{'no LN':>11s}{'LN':>11s}" * len(cols))
for mesh in LN_MESHES:
    print(f"{mesh:10s}" + "".join(f"{float(ln_errors[f'{c}__{mesh}__{metric}_{mode}']):>11.3e}"
                                  for mode, metric in cols for c in pair))
print(f"{'mean':10s}" + "".join(
    f"{np.mean([float(ln_errors[f'{c}__{mesh}__{metric}_{mode}']) for mesh in LN_MESHES]):>11.3e}"
    for mode, metric in cols for c in pair))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# LayerNorm is the variant under test (solid, PLOT1) and the plain 128-channel
# residual run the baseline (dashed, PLOT2), the roles msg_dim = 1 / 128 take
# above. Shared y-axis, so the training and validation panels read on one scale.
ln_styles = {"LayerNorm": (PLOT1_COLOR, "-"), "no LayerNorm": (PLOT2_COLOR, "--")}

fig, axs = plt.subplots(1, 2, figsize=(1400*px, 540*px), sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Loss Decay, LayerNorm vs none, Residual FV, msg_dim = 128, h = 1",
             fontsize=24, color=AX_COLOR)

for ax, (title, losses) in zip(axs, (("Training Loss", ln_train_loss),
                                     ("Validation Loss", ln_val_loss))):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, (color, ls) in ln_styles.items():
        ax.plot(losses[key], label=key, color=color, linewidth=2.5, linestyle=ls)

    # Value where each curve ends: the higher one labelled above its line and
    # the lower one below, so the two labels cannot collide.
    ends = sorted(((losses[k][-1], k) for k in ln_styles), reverse=True)
    for (y, key), dy, va in zip(ends, (8, -8), ("bottom", "top")):
        ax.annotate(f"{y:.1e}", xy=(len(losses[key]) - 1, y), xytext=(0, dy),
                    textcoords="offset points", ha="right", va=va,
                    fontsize=14, color=AX_COLOR)

    ax.set_yscale("log")

    legend = ax.legend(fontsize=18)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if ax is axs[0]:
        ax.set_ylabel("MSE Loss", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_layernorm_loss.png", dpi=300, transparent = True)

In [ ]:
# --- Message vs FV flux: R^2 only, LayerNorm vs plain --------------------------
# compute_model_flux_stats from the residual cell, on the same meshes, timesteps
# and seeded faces, so every row lines up with the plain 128-channel model's
# (flux_resid). Only the R^2 read-outs are printed: F, dT and w each regressed on
# all 128 channels, sum_j m_ij against the net flux at node level, and the same
# three edge read-outs restricted to the near-orthogonal faces.
flux_ln_cache = processed_data_dir / "parametric_message_flux_layernorm.npz"

if flux_ln_cache.exists():
    flux_ln = dict(np.load(flux_ln_cache, allow_pickle=False))
else:
    flux_ln = compute_model_flux_stats(LN_EXP)
    np.savez_compressed(flux_ln_cache, **flux_ln)

flux_lnc = {**flux_resid, **flux_ln}
print(f"R^2 of the linear read-out from the message, {N_STEPS} probed timesteps")
print(f"{'mesh':<11}{'model':<14}{'F':>7}{'dT':>7}{'w':>7}{'node':>7}   F/dT/w on cos>0.99")
for mesh in FLUX_MESHES:
    for label, exp in LN_CASES.items():
        print(f"{mesh:<11}{label:<14}"
              + "".join(f"{v:>7.3f}" for v in flux_lnc[f"{exp}__{mesh}__r2"])
              + f"{float(flux_lnc[f'{exp}__{mesh}__r2_node']):>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux_lnc[f"{exp}__{mesh}__r2_near"]))